# 📉 Customer Churn Prediction

**Dataset**: IBM Telco Customer Churn — 7,043 rows × 21 columns  
**Target**: `Churn` (Yes / No) → Binary Classification  

---
## Section 1 — Data Exploration
## Section 2 — Data Cleaning & Preprocessing

---
## Section 1 — Data Exploration

In [ ]:
import sys, os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.config import RAW_DATA_PATH, TARGET_COL, NUMERICAL_COLS, PLOT_STYLE, FIG_DPI, COLOR_CHURN, IMAGES_DIR

pd.set_option('display.max_columns', 30)
plt.style.use(PLOT_STYLE)
print('Libraries ready')

In [ ]:
# Load raw dataset
df = pd.read_csv(RAW_DATA_PATH)
print(f'Shape: {df.shape}')
df.head()

In [ ]:
# Data types and non-null counts
df.info()

In [ ]:
# Statistical summary — numerical
df.describe()

In [ ]:
# Statistical summary — categorical
df.describe(include='object')

In [ ]:
# Standard null check
print('NaN counts:')
print(df.isnull().sum())

# Hidden blank strings (TotalCharges issue)
print('\nHidden blank strings in object columns:')
for col in df.select_dtypes(include='object').columns:
    n = (df[col].str.strip() == '').sum()
    if n > 0:
        print(f'  [{col}]: {n} blank value(s)')

In [ ]:
# Duplicate check
print(f'Duplicate rows: {df.duplicated().sum()}')
print(f'Unique customerIDs: {df["customerID"].nunique()} (total rows: {len(df)})')

In [ ]:
# Target variable distribution
print(df[TARGET_COL].value_counts())
print(df[TARGET_COL].value_counts(normalize=True) * 100)

In [ ]:
# Churn distribution — bar + pie
churn_counts = df[TARGET_COL].value_counts()
churn_pct    = df[TARGET_COL].value_counts(normalize=True) * 100

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Customer Churn Distribution', fontsize=15, fontweight='bold')
colors = [COLOR_CHURN['No'], COLOR_CHURN['Yes']]

# Bar
bars = axes[0].bar(churn_counts.index, churn_counts.values, color=colors, edgecolor='white', width=0.5)
axes[0].set_title('Count'); axes[0].set_ylabel('Customers')
for bar, (lbl, cnt, pct) in zip(bars, zip(churn_counts.index, churn_counts.values, churn_pct.values)):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 40,
                 f'{cnt:,}\n({pct:.1f}%)', ha='center', fontsize=11, fontweight='bold')
axes[0].set_ylim(0, churn_counts.max() * 1.2)

# Pie
axes[1].pie(churn_counts.values, labels=churn_counts.index, colors=colors,
            autopct='%1.1f%%', startangle=90, explode=(0, 0.05),
            wedgeprops=dict(edgecolor='white', linewidth=2))
axes[1].set_title('Proportion')

plt.tight_layout()
plt.savefig(IMAGES_DIR / 'churn_distribution.png', dpi=FIG_DPI, bbox_inches='tight')
plt.show()

In [ ]:
# Numerical features — histograms & boxplots by Churn
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('Numerical Feature Distributions', fontsize=15, fontweight='bold')

for i, col in enumerate(NUMERICAL_COLS):
    col_data = pd.to_numeric(df[col], errors='coerce').dropna()

    # Histogram
    axes[0][i].hist(col_data, bins=30, color='steelblue', edgecolor='white', alpha=0.85)
    axes[0][i].set_title(f'{col} — Histogram'); axes[0][i].set_xlabel(col)

    # Boxplot by Churn
    tmp = df[['Churn', col]].copy()
    tmp[col] = pd.to_numeric(tmp[col], errors='coerce')
    groups = [tmp[tmp['Churn'] == lbl][col].dropna() for lbl in ['No', 'Yes']]
    bp = axes[1][i].boxplot(groups, labels=['No (Stayed)', 'Yes (Churned)'], patch_artist=True)
    for patch, c in zip(bp['boxes'], [COLOR_CHURN['No'], COLOR_CHURN['Yes']]):
        patch.set_facecolor(c); patch.set_alpha(0.7)
    axes[1][i].set_title(f'{col} — by Churn')

plt.tight_layout()
plt.savefig(IMAGES_DIR / 'numerical_distributions.png', dpi=FIG_DPI, bbox_inches='tight')
plt.show()

In [ ]:
# Churn rate per key categorical features
cols_to_plot = ['gender', 'SeniorCitizen', 'Partner', 'Dependents',
                'Contract', 'InternetService', 'PaymentMethod', 'PaperlessBilling']

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
fig.suptitle('Churn Rate by Categorical Features', fontsize=15, fontweight='bold')

for idx, col in enumerate(cols_to_plot):
    ax = axes[idx // 4][idx % 4]
    tmp = df[[col, TARGET_COL]].copy()
    tmp[TARGET_COL] = tmp[TARGET_COL].map({'No': 0, 'Yes': 1})
    churn_rate = tmp.groupby(col)[TARGET_COL].mean().sort_values(ascending=False) * 100
    bars = ax.bar(churn_rate.index.astype(str), churn_rate.values,
                  color='#E57373', edgecolor='white', alpha=0.85)
    ax.set_title(col, fontsize=11, fontweight='bold')
    ax.set_ylabel('Churn Rate (%)')
    ax.set_xticklabels(churn_rate.index.astype(str), rotation=30, ha='right', fontsize=8)
    ax.axhline(y=26.5, color='gray', linestyle='--', linewidth=1, alpha=0.7)
    for bar, val in zip(bars, churn_rate.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                f'{val:.1f}%', ha='center', fontsize=8, fontweight='bold')

plt.tight_layout()
plt.savefig(IMAGES_DIR / 'categorical_churn_rates.png', dpi=FIG_DPI, bbox_inches='tight')
plt.show()

**Key Findings — Exploration**
- 7,043 rows × 21 columns, no true NaN values
- `TotalCharges` stored as string with 11 hidden blank rows (tenure=0 customers)
- Class imbalance: 73.5% stayed / 26.5% churned → use F1 & ROC-AUC as metrics
- Top signals: `Contract` type, `InternetService`, `tenure`, `MonthlyCharges`, `PaymentMethod`

---
## Section 2 — Data Cleaning & Preprocessing

In [ ]:
from src.preprocessing import load_raw_data, clean_data, get_cleaning_report
from src.config import CLEANED_DATA_PATH

df_raw = load_raw_data(RAW_DATA_PATH)

In [ ]:
# Inspect the 11 blank TotalCharges rows (tenure=0 customers, never billed)
problematic = df_raw[df_raw['TotalCharges'].str.strip() == '']
print(f'Rows with blank TotalCharges: {len(problematic)}')
problematic[['customerID', 'tenure', 'MonthlyCharges', 'TotalCharges', 'Churn']].reset_index(drop=True)

In [ ]:
# Run full cleaning pipeline
# Steps: fix TotalCharges → drop customerID → remove duplicates → validate SeniorCitizen → encode Churn
df_clean = clean_data(df_raw, id_col='customerID', target_col='Churn', encode_target_flag=True)

In [ ]:
# Before vs After report
_ = get_cleaning_report(df_raw, df_clean)

In [ ]:
df_clean.head()

In [ ]:
# Verify cleaning — 3-panel plot
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Cleaning Verification', fontsize=14, fontweight='bold')

# 1. TotalCharges distribution after fix
axes[0].hist(df_clean['TotalCharges'], bins=40, color='#5C6BC0', edgecolor='white', alpha=0.85)
axes[0].axvline(df_clean['TotalCharges'].median(), color='#FF7043', linewidth=2, linestyle='--',
                label=f'Median={df_clean["TotalCharges"].median():.0f}')
axes[0].axvline(df_clean['TotalCharges'].mean(), color='#26C6DA', linewidth=2, linestyle='--',
                label=f'Mean={df_clean["TotalCharges"].mean():.0f}')
axes[0].set_title('TotalCharges (After Fix)'); axes[0].legend(fontsize=9)

# 2. Null heatmap
null_pct = (df_clean.isnull().sum() / len(df_clean) * 100).reset_index()
null_pct.columns = ['Column', 'Null%']
bar_colors = ['#EF5350' if v > 0 else '#66BB6A' for v in null_pct['Null%']]
axes[1].barh(null_pct['Column'], null_pct['Null%'], color=bar_colors)
axes[1].set_title('Missing Values After Cleaning'); axes[1].set_xlabel('Missing %')
axes[1].text(0.5, 0.98, '✅ All zeros!', transform=axes[1].transAxes,
             ha='center', va='top', fontsize=12, color='green', fontweight='bold')

# 3. Encoded target
cc = df_clean[TARGET_COL].value_counts().sort_index()
bars = axes[2].bar(['No Churn (0)', 'Churn (1)'], cc.values,
                   color=[COLOR_CHURN['No'], COLOR_CHURN['Yes']], edgecolor='white', width=0.5)
for bar, cnt in zip(bars, cc.values):
    axes[2].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 40,
                 f'{cnt:,}\n({cnt/len(df_clean)*100:.1f}%)', ha='center', fontsize=11, fontweight='bold')
axes[2].set_title('Churn Target (0/1 Encoded)'); axes[2].set_ylim(0, cc.max() * 1.2)

plt.tight_layout()
plt.savefig(IMAGES_DIR / 'cleaning_verification.png', dpi=FIG_DPI, bbox_inches='tight')
plt.show()

In [ ]:
# Correlation heatmap — numerical features + target
corr = df_clean[NUMERICAL_COLS + [TARGET_COL]].corr()
fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(corr, annot=True, fmt='.3f', cmap='RdYlGn', center=0,
            vmin=-1, vmax=1, linewidths=0.5, square=True, ax=ax)
ax.set_title('Correlation Matrix — Numerical + Target', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(IMAGES_DIR / 'correlation_heatmap.png', dpi=FIG_DPI, bbox_inches='tight')
plt.show()

print('\nCorrelation with Churn:')
print(corr[TARGET_COL].drop(TARGET_COL).sort_values(key=abs, ascending=False))

In [ ]:
# Save cleaned dataset
df_clean.to_csv(CLEANED_DATA_PATH, index=False)
print(f'Saved → {CLEANED_DATA_PATH}')
print(f'Shape : {df_clean.shape[0]:,} rows × {df_clean.shape[1]} columns')
print(f'Nulls : {df_clean.isnull().sum().sum()}')

**Key Findings — Cleaning**
- `TotalCharges`: converted to float, 11 blank rows imputed with median (right-skewed → median preferred over mean)
- `customerID`: dropped (identifier, no predictive signal)
- Duplicates: 0 found
- `Churn`: encoded Yes→1, No→0
- Final shape: 7,043 rows × 20 columns, 0 nulls
- `TotalCharges` and `tenure` are highly correlated → will engineer `AvgMonthlySpend` in Feature Engineering